In [0]:
# Databricks Notebook: 02_Silver_Clean
# Silver 层：bronze → silver 清洗、去重、类型标准化
# 幂等策略：
#   - silver.mes（缓慢变化）：MERGE INTO + row_number() 去重取最新
#   - 其他事实表（不可变）：DELETE 按天 → APPEND
# 约定：必须 from pyspark.sql.window import Window（F.partitionBy 会 AttributeError）

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
print(f"📅 Silver 清洗日期: {today}")

# =====================================================
# 1. silver.mes：MES 工单（有 UPDATE 场景 → MERGE INTO）
#    面试考点：hold 批次后续变 completed = 更新 → 用 MERGE 而非 APPEND
# =====================================================
print("=" * 50)
print("1. silver.mes（MERGE INTO 增量更新）")
print("=" * 50)

# 1.1 去重：同一 lot 多条记录取 ingest_ts 最新的一条
latest_mes = (spark.table("bronze.raw_mes")
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("lot_id")
              .orderBy(F.col("ingest_ts").desc(), F.col("start_date").desc())))
    .filter("rn = 1")
    .drop("rn", "ingest_date", "ingest_ts")
    .createOrReplaceTempView("mes_latest"))

# 1.2 首次运行建表（Liquid Clustering）
if not spark.catalog.tableExists("silver.mes"):
    spark.sql("""
        CREATE TABLE silver.mes CLUSTER BY (lot_id) AS
        SELECT * FROM mes_latest WHERE 1 = 0
    """)
    print("🆕 已创建 silver.mes (CLUSTER BY lot_id)")

# 1.3 MERGE INTO：新批次插入，老批次状态更新
spark.sql("""
    MERGE INTO silver.mes AS t
    USING mes_latest AS s
    ON t.lot_id = s.lot_id
    WHEN MATCHED AND t.status <> s.status THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
total = spark.table("silver.mes").count()
print(f"✅ silver.mes MERGE 完成，当前 {total} 行")

# =====================================================
# 2. 事实表通用清洗：DELETE 按天 → APPEND（不可变数据）
# =====================================================
print("=" * 50)
print("2. 事实表清洗（按天幂等）")
print("=" * 50)

# --- 2.1 silver.moves 过站明细 ---
clean_moves = (spark.table("bronze.raw_mes_moves")
    .filter(F.col("qty_out") >= 0)                                    # 异常数量剔除
    .withColumn("track_in_ts", F.to_timestamp("track_in_ts"))
    .withColumn("track_out_ts", F.to_timestamp("track_out_ts")))
if not spark.catalog.tableExists("silver.moves"):
    (clean_moves.limit(0).write.format("delta")
        .clusterBy("move_date", "lot_id").saveAsTable("silver.moves"))
    print("🆕 已创建 silver.moves (CLUSTER BY move_date, lot_id)")
spark.sql(f"DELETE FROM silver.moves WHERE move_date = '{today}'")
(clean_moves.write.mode("append").format("delta").saveAsTable("silver.moves"))
print(f"✅ silver.moves: 当天 {clean_moves.count()} 行")

# --- 2.2 silver.cp_bins CP 测试 ---
clean_cp = (spark.table("bronze.raw_cp_bins")
    .filter((F.col("die_count") >= 0) & F.col("cp_yield_pct").between(0, 100)))
if not spark.catalog.tableExists("silver.cp_bins"):
    (clean_cp.limit(0).write.format("delta")
        .clusterBy("test_date", "lot_id").saveAsTable("silver.cp_bins"))
spark.sql(f"DELETE FROM silver.cp_bins WHERE test_date = '{today}'")
(clean_cp.write.mode("append").format("delta").saveAsTable("silver.cp_bins"))
print(f"✅ silver.cp_bins: 当天 {clean_cp.count()} 行")

# --- 2.3 silver.lot_events 批次事件 ---
clean_events = spark.table("bronze.raw_lot_events") \
    .withColumn("event_ts", F.to_timestamp("event_ts"))
if not spark.catalog.tableExists("silver.lot_events"):
    (clean_events.limit(0).write.format("delta")
        .clusterBy("event_date").saveAsTable("silver.lot_events"))
spark.sql(f"DELETE FROM silver.lot_events WHERE event_date = '{today}'")
(clean_events.write.mode("append").format("delta").saveAsTable("silver.lot_events"))
print(f"✅ silver.lot_events: 当天 {clean_events.count()} 行")

# --- 2.4 silver.equip_state 设备状态（OEE 数据源）---
clean_state = spark.table("bronze.raw_equip_state") \
    .withColumn("start_ts", F.to_timestamp("start_ts")) \
    .withColumn("end_ts", F.to_timestamp("end_ts")) \
    .filter(F.col("duration_min") > 0)
if not spark.catalog.tableExists("silver.equip_state"):
    (clean_state.limit(0).write.format("delta")
        .clusterBy("state_date", "equipment_id").saveAsTable("silver.equip_state"))
spark.sql(f"DELETE FROM silver.equip_state WHERE state_date = '{today}'")
(clean_state.write.mode("append").format("delta").saveAsTable("silver.equip_state"))
print(f"✅ silver.equip_state: 当天 {clean_state.count()} 行")

# --- 2.5 silver.quality 质检 ---
clean_qc = (spark.table("bronze.raw_quality")
    .filter(F.col("yield_rate").between(0, 100)))
if not spark.catalog.tableExists("silver.quality"):
    (clean_qc.limit(0).write.format("delta")
        .clusterBy("inspection_date").saveAsTable("silver.quality"))
spark.sql(f"DELETE FROM silver.quality WHERE inspection_date = '{today}'")
(clean_qc.write.mode("append").format("delta").saveAsTable("silver.quality"))
print(f"✅ silver.quality: 当天 {clean_qc.count()} 行")

# =====================================================
# 3. 验证
# =====================================================
print("=" * 50)
print("Silver 层总行数")
print("=" * 50)
for t in ["mes", "moves", "cp_bins", "lot_events", "equip_state", "quality"]:
    print(f"  silver.{t}: {spark.table(f'silver.{t}').count()}")
print("\n🎉 Silver 清洗完成！")


📅 Silver 清洗日期: 2026-09-03
1. silver.mes（MERGE INTO 增量更新）
✅ silver.mes MERGE 完成，当前 202 行
2. 事实表清洗（按天幂等）
✅ silver.moves: 当天 1209 行
✅ silver.cp_bins: 当天 0 行
✅ silver.lot_events: 当天 21 行
✅ silver.equip_state: 当天 277 行
✅ silver.quality: 当天 200 行
Silver 层总行数
  silver.mes: 202
  silver.moves: 1209
  silver.cp_bins: 0
  silver.lot_events: 21
  silver.equip_state: 277
  silver.quality: 200

🎉 Silver 清洗完成！
